# 06 · The loop runs itself

*Part 3.* Every step from notebooks 01-05 is a **workflow**: a function that runs as one MLflow
run with its outcome, metrics and artifacts, callable from Python, the `selfimprove` CLI, or a
scheduler. `cycle()` chains them:

```
score -> monitor -> align -> triage -> approve -> [enough failures?] -> improve -> gate
```

| step | scheduled | optimizer | stays with a person |
|---|---|---|---|
| reading the themes | triage with a cap, contradiction path, provenance | | contradictions |
| keeping signal flowing | sampled evaluation | | golden cases, expectations |
| keeping the judge aligned | nightly agreement check | `align()` | the threshold |
| approving the tests | approver checks, then merge | | flagged tests, audit sample, criteria |
| changing the prompt | candidate scored on CI + holdout | `optimize_prompts` | the promotion margin |
| changing a skill or tool | | | accepting the proposal |
| promoting | alias moves when both gates pass | | rollback review |

In [ ]:
import _nb as nb

experiment_id = nb.setup()

In [ ]:
import os
from selfimprove import agents, config, instructions, store
from selfimprove.workflows.cycle import cycle
from selfimprove.workflows.report import report

os.environ["SI_AUTO_PROMOTE"] = "true"     # the gate presses the button in this notebook
os.environ["SI_IMPROVER"] = "analyst"           # the LLM trace analyst writes the candidate
config.reload()
print("production is v" + str(instructions.load("production").version))

## 1. New traffic arrives

A mix of the usual questions and the new support-process ones, then a round of labels.

In [ ]:
from examples import reviewer

regular = [q for q in open("examples/questions.txt").read().splitlines() if q and not q.startswith("#")]
drift = [q for q in open("examples/drift.txt").read().splitlines() if q and not q.startswith("#")]
for q in regular + drift * 2:
    agents.run(q, source="production")
reviewer.review_all(sample=0.75)

## 2. One cycle

Each nested run in the MLflow UI is one job. The cycle stops early when there is nothing to do
(`waiting`, `healthy`), when the judge cannot be trusted (`halted_untrusted_judge`, which queues
it for a person), or when the gate refuses the candidate.

In [ ]:
result = cycle(force=True)
print()
print(result)

**Under the hood:** `cycle()` opens a run with `si.job=cycle`. Each job inside it calls
`mlflow.start_run(..., nested=True)` because a run is already active, so the jobs appear as child runs,
each with its own `si.job`, `si.outcome`, `si.message`, params and metrics. `score`'s
`mlflow.genai.evaluate()` call logs into the `score` run. A `gate` job adds one more level: the
`gate: vN vs vM` run and its four evaluation runs.

In [ ]:
import mlflow

cyc = mlflow.search_runs(filter_string="tags.`si.job` = 'cycle'", order_by=["start_time DESC"],
                        max_results=1, output_format="list")[0]
print(cyc.info.run_name, "->", cyc.data.tags.get("si.outcome"))
for r in mlflow.search_runs(filter_string=f"tags.mlflow.parentRunId = '{cyc.info.run_id}'",
                            order_by=["start_time ASC"], output_format="list"):
    print(f"  {r.data.tags.get('si.job'):<8} {r.data.tags.get('si.outcome', ''):<16} {r.data.tags.get('si.message', '')[:70]}")

In [ ]:
prod = instructions.load("production")
print(f"production is v{prod.version} (promoted by {prod.tags.get('promoted_by', 'seed')})")
print(instructions.diff("previous", "production") if prod.tags.get("promoted_by") == "gate" else "")

## 3. Live traffic decides whether it stays

After a gate promotion, `monitor` compares the new version's live failure rate with the rate it
replaced, and rolls back if it rose by more than `SI_ROLLBACK_MARGIN`. A promotion is always one
alias move away from being undone.

In [ ]:
from selfimprove.workflows.ship import monitor

for q in regular + drift:
    agents.run(q, source="production")
from selfimprove.workflows.score import score
score()
print(monitor())

**Under the hood:** `monitor` reads the `@production` prompt version's tags (`promoted_by`,
`promoted_at`, `replaced_failure_rate`) and the live failure rate from
`mlflow.search_traces(filter_string="tags.si_source = 'production' AND tags.si_prompt_version = '<v>'")`.
It only acts on versions the gate promoted. A rollback is `mlflow.genai.set_prompt_alias(name,
"production", <replaced version>)` plus version tags `status=rolled_back` and `rolled_back_because`.
`score` and `triage` remember where they stopped in experiment tags (`si.checkpoint.<job>`).

In [ ]:
print({k: v for k, v in mlflow.get_experiment(experiment_id).tags.items() if k.startswith("si.")})
for v in mlflow.MlflowClient().search_prompt_versions(config.settings.prompt_name):
    p = mlflow.genai.load_prompt(f"prompts:/{config.settings.prompt_name}/{v.version}")
    print(f"v{p.version}  aliases={p.aliases}  status={p.tags.get('status')}  promoted_by={p.tags.get('promoted_by', '-')}")

In [ ]:
print(report())

## 4. Scheduling it

From a terminal, each job is one command (`selfimprove score`, `triage`, `approve`, `align`, `improve`,
`gate`, `monitor`, `cycle`, `report`). `.github/workflows/selfimprove-loop.yml` runs the nightly
jobs and a weekly cycle against a shared MLflow server, off until you set
`SELFIMPROVE_ENABLED=true`, and with promotion left to a person unless `SI_AUTO_PROMOTE=true`.

What stays with a person is what defines "good": the thresholds and criteria, golden cases and
expectations, the queue, an audit sample of what the approver merged, any change to skills, tools
or code, and rollback review. Fully autonomous self-evolution is, in Gao et al.'s words, "an
aspirational goal rather than the current norm": the gap is in verification more than tooling.